# Inspect MLX checkpoint continuations

Run this notebook from the repo root with the `.venv` kernel (or adjust `run_dir`).
This is a next-token model trained on web text: give it sentence beginnings, rather
than chat instructions. Load once, then generate short samples manually.

Compare readable words, sentence structure, topic consistency, repetition, and
whether the continuation follows the prompt. These samples are qualitative;
they do not measure held-out performance or establish memorization.

Lower temperature sharpens the sampling distribution; higher temperature flattens
it. Keep seed and prompt fixed when comparing temperatures. EOS is displayed as
`<|endoftext|>` and ends generation. Context is cropped to the trained length.

Implementation: `src/llm_training/models/gpt_mlx.py` and
`src/llm_training/pretraining_mlx/generate.py`.
Reference: [MLX categorical sampling](https://ml-explore.github.io/mlx/build/html/python/random/_autosummary/mlx.core.random.categorical.html).


In [ ]:
from pathlib import Path
from llm_training.pretraining_mlx.generate import load_checkpoint, generate_text

run_dir = Path("artifacts/pretraining/tiny-gpt-mlx-run-03")
model, tokenizer, eos_id = load_checkpoint(run_dir)
print(f"Context: {model.config.context_length}; vocabulary: {model.config.vocab_size}")


In [ ]:
prompt = "Сегодня утром я"
print(generate_text(model, tokenizer, prompt, max_new_tokens=100,
                    temperature=0.8, seed=42, eos_id=eos_id))


In [ ]:
# Run when ready to compare the same sentence beginning at three temperatures.
for temperature in (0.5, 0.8, 1.0):
    print(f"\nTemperature: {temperature}")
    print(generate_text(model, tokenizer, prompt, max_new_tokens=100,
                        temperature=temperature, seed=42, eos_id=eos_id))


Try changing `prompt` to `"В Москве сегодня"` or `"Для приготовления супа"`,
then rerun the single-sample cell. Change the seed to check how much conclusions
about quality depend on a particular draw. Optionally load another run to compare;
check its config before attributing differences to any one training choice.
